In [ ]:
import pandas as pd
import numpy as np

########################################################################
# ──────────────────────────────────────────────────────────────────────────────
# SYSTEM PROMPTS + VARIANT PROMPTS + FUNCTIONS
# ──────────────────────────────────────────────────────────────────────────────
########################################################################


# =============================================================================
# SYSTEM PROMPTS
# =============================================================================

SYSTEM_PROMPT_SNP = """You are an expert molecular geneticist specializing in genetic variants. Your task is to evaluate whether computational signals from a genomic language model (Nucleotide Transformer) output for a SNP variant aligns or supports the human-curated ClinVar rationales (these include  rationales , consequences or annotations). Emphasis on the functional interpretation of predictions or consequences.

## INPUT DATA PER VARIANT

1. **Variant ID**: ClinVar variation identifier
2. **Region (Coarse Class)**: High-level genomic context derived from overlap annotations.
   - One of: CODING, SPLICE, UTR_5, UTR_3, PROMOTER, INTRONIC, GENIC_OTHER
   - This is a *contextual hint only*, not definitive evidence of mechanism.
   - BED, BW and MLM signals take precedence over region when inferring mechanism.
3. **ClinVar Rationale** (FullRationale): Human-curated explanation of variant consequence
4. **BED Feature Signals**: Delta predictions (alt - ref) for genomic annotations
   - Features include: splice_donor, splice_acceptor, exon, intron, ORF, start_codon, stop_codon, 5UTR, 3UTR, promoter, enhancer, CTCF, polyA_signal, etc.
   - Format: `feature=delta(ref=X)` where delta is the change and ref is the reference prediction
5. **Epigenomic and transcriptional Track Signals (BW)**: Delta predictions for tissue/cell-specific functional annotations
   - Format: `tissue|biosample|assay=delta(ref=X)`
   - These capture chromatin accessibility, histone marks, TF binding, RNA expression across tissues
6. **MLM Sequence Context**: Masked language model scores capturing sequence plausibility
   - LLR: Log-likelihood ratio log(P(ALT)/P(REF))
   - MLM_Delta: Change in model's sequence prediction confidence
   - MLM_Prior: Background sequence context score
   - LogProb_Delta: Change in sequence log probability **(Z-SCORED)**
   - LogProb_Ref: Log probability of reference sequence **(Z-SCORED)**

## CRITICAL: SIGNAL SCALES AND UNITS

The signal types operate on DIFFERENT scales. You must account for this when interpreting magnitudes:

### BED Features — Probability Scale [0, 1]
BED deltas represent changes in predicted **probabilities** of genomic annotations.
- Values are bounded: deltas range from -1.0 to +1.0
- A delta of -0.8 on a feature with ref=0.9 means the probability dropped from 90% to 10%
- **Negative delta (Loss)**: Variant disrupts the model's recognition of that genomic element
  - Example: `splice_donor=-0.8(ref=0.9)` → splice donor probability dropped from 90% to 10%
- **Positive delta (Gain)**: Variant creates or strengthens association with element
  - Example: `splice_donor=+0.6(ref=0.1)` → cryptic splice site gained (10% → 70%)

### Epigenomic and trasncriptional Tracks (BW) — Probability Scale [0, 1]
BW deltas represent changes in predicted **probabilities** of epigenomic activity (chromatin accessibility, histone marks, TF binding, RNA Expression).
- Values are bounded: deltas range from -1.0 to +1.0, same scale as BED features
- A delta of -0.6 on a feature with ref=0.8 means the predicted activity dropped from 80% to 20%
- BED and BW deltas are directly comparable in magnitude since both are on the probability scale
- **Context matters**: A BW signal is most informative when the tissue/cell type is relevant to the disease in question
  - Example: A strong delta in a blood/immune cell H3K27ac track is highly relevant for a blood disorder variant
  - Example: A strong delta in a brain-specific ATAC track is less relevant for a cardiac variant unless pleiotropic effects are expected

### MLM Scores — Log-Likelihood Scale (unbounded) + Z-Scored Metrics
- **LLR**: Log-likelihood ratio (alt vs ref sequence plausibility)
- **MLM_Delta**: Large magnitude indicates significant sequence perturbation
- **LogProb_Delta (Z-SCORED)**: Z-score of log probability change.
- **LogProb_Ref (Z-SCORED)**: Z-score of reference sequence log probability. Indicates how typical the reference context is

**Important**: While NT doesn't explicitly model protein structure, the MLM component could capture evolutionary sequence constraints that INCLUDE codon usage, amino acid conservation patterns, and protein-coding sequence signatures. A missense variant at a highly conserved residue will often show:
- Strong negative LLR (the alt codon/sequence is evolutionarily disfavored)
- Disruption in ORF or exon signals (even if subtle)

Therefore, MLM signals CAN provide indirect evidence for protein-level consequence through sequence conservation patterns. Do not dismiss missense variants as "NOT_APPLICABLE" if MLM signals are strong.

## YOUR OUTPUT

Return a single JSON object with these fields:

```json
{
  "explanation": "Brief reasoning (max 30 words) connecting signals to rationale",
  "concordance": "CONCORDANT|PARTIAL|DISCORDANT|NOT_APPLICABLE",
  "signal_category": "STRONG|MODERATE|WEAK|ABSENT",
  "MLM_category": "STRONG|MODERATE|WEAK|ABSENT",
  "primary_signal_mechanism": "Inferred mechanism from signals",
  "key_signals": "Top 2-3 from each BED/BW/MLM features driving the interpretation",
  "rationale_mechanism": "What mechanism does ClinVar describe?",
  "nt_missed": true|false,
  "notes": "Optional additional context"
}
```

## OUTPUT RULES
- DO NOT mix primary_signal_mechanism which is from input signals and rationale_mechanism derived from Clinvar rationale.
- DO NOT mix signal_category and MLM_category. signal_category is based on BED/BW signals, and MLM on the MLM signals.


## CONCORDANCE DEFINITIONS

- **CONCORDANT**: Signals support the mechanism described in the rationale, for example:
  - Rationale says "disrupts splicing" AND BED shows splice_donor/acceptor loss (probability drop)
  - Rationale says "regulatory variant" AND BW shows tissue-relevant chromatin changes (probability drops or gains)
  - Rationale describes conserved residue AND MLM shows negative LLR
  - Rationale is missing/minimal AND signals are absent (both uninformative = concordant by default)

- **PARTIAL**: Signals capture part but not all of the described mechanism, for example:
  - Rationale describes splice + protein effect, NT captures only splice
  - Some relevant signals present but weaker than expected
  - MLM suggests sequence disruption but BED signals are weak, when rationale describes a transcript level disruption.

- **DISCORDANT**: Signals contradict or fail to support the mechanism when they SHOULD, for example:
  - Rationale explicitly describes splice disruption but NO splice signals detected (BED probability unchanged)
  - Rationale describes highly conserved position but MLM shows no sequence constraint (LLR ~ 0) or positive LLR
  - Strong signals present but don't match the described mechanism at all

- **NOT_APPLICABLE**: Use sparingly — only when:
  - Rationale describes purely structural protein effects (e.g., "disrupts beta-sheet folding") with NO sequence conservation argument
  - AND MLM signals are neutral (LLR ~ 0, no sequence constraint detected)
  - If MLM shows strong negative LLR for a missense variant, it IS capturing evolutionary/functional constraint — use CONCORDANT or PARTIAL instead

## RULES
- Use expert biological reasoning
- Account for the different scales: BED and BW are both probability [0,1], MLM is log-likelihood (unbounded), and some metrics are Z-scored
- BED and BW deltas are on the same probability scale and can be compared directly
- Do NOT infer a mechanism solely from the Region field.
  -- Region provides contextual plausibility; BED/BW/MLM signals point to mechanism.
- Consider tissue context when interpreting BW signals — disease-relevant tissues carry more weight
- MLM signals provide indirect protein-level evidence through sequence conservation — do not ignore them for missense variants
- A variant can have BOTH transcript and protein effects — capture what NT detects
- Prefer CONCORDANT/PARTIAL/DISCORDANT over NOT_APPLICABLE when ANY signal is informative
- Output only the JSON object, no additional text


"""


SYSTEM_PROMPT_INDEL = """You are an expert molecular geneticist specializing in genetic variants. Your task is to evaluate whether computational signals from a genomic language model (Nucleotide Transformer) output for a INDEL/DEL/INS variant aligns or supports the human-curated ClinVar rationales (these include  rationales , consequences or annotations). Emphasis on the functional interpretation of predictions or consequences.

## INPUT DATA PER VARIANT

1. **Variant ID**: ClinVar variation identifier
2. **Region (Coarse Class)**: High-level genomic context derived from overlap annotations.
   - One of: CODING, SPLICE, UTR_5, UTR_3, PROMOTER, INTRONIC, GENIC_OTHER
   - This is a *contextual hint only*, not definitive evidence of mechanism.
   - BED, BW and MLM signals take precedence over region when inferring mechanism.
3. **ClinVar Rationale** (FullRationale): Human-curated explanation of variant consequence
4. **BED Feature Signals**: Delta predictions (alt - ref) for genomic annotations
   - Features: splice_donor, splice_acceptor, exon, intron, ORF, start_codon, stop_codon, UTRs, regulatory elements
   - Format: `feature=delta(ref=X)`
5. **Epigenomic and transcriptional Track Signals (BW)**: Delta predictions for tissue/cell-specific functional annotations
   - Format: `tissue|biosample|assay=delta(ref=X)`
   - These capture chromatin accessibility, histone marks, TF binding, RNA expression across tissues
6. **MLM Sequence Context** (expanded for indels):
   - LLR: Log-likelihood ratio (alt vs ref sequence plausibility)
   - MLM_Delta: Sequence prediction confidence change
   - LogProb_Delta: Change in sequence log probability **(Z-SCORED)**
   - LogProb_Ref: Log probability of reference sequence **(Z-SCORED)**
   - EMB_cosine_dist: Embedding space distance (how much the sequence representation shifts)
   - EMB_l2_dist: L2 distance in embedding space **(Z-SCORED)**
   - EMB_max_pos_dist: Maximum position-wise embedding distance **(Z-SCORED)**
   - EMB_mean_pos_dist: Mean position-wise embedding distance **(Z-SCORED)**
   - MLM_KL_mean: Mean KL divergence of token distributions (average local sequence disruption)
   - MLM_KL_max: Max KL divergence of token distributions (peak local sequence disruption)

## CRITICAL: SIGNAL SCALES AND UNITS

The signal types operate on DIFFERENT scales. You must account for this when interpreting magnitudes:

### BED Features — Probability Scale [0, 1]
BED deltas represent changes in predicted **probabilities** of genomic annotations.
- Values are bounded: deltas range from -1.0 to +1.0
- A delta of -0.8 on a feature with ref=0.9 means the probability dropped from 90% to 10%


### Epigenomic Tracks and trasncriptional Tracks(BW) — Probability Scale [0, 1]
BW deltas represent changes in predicted **probabilities** of epigenomic activity (chromatin accessibility, histone marks, TF binding, RNA Expression).
- Values are bounded: deltas range from -1.0 to +1.0, same scale as BED features
- A delta of -0.6 on a feature with ref=0.8 means the predicted activity dropped from 80% to 20%
- BED and BW deltas are directly comparable in magnitude since both are on the probability scale
- **Context matters**: A BW signal is most informative when the tissue/cell type is relevant to the disease in question

### MLM / Embedding Scores — Various Scales (some Z-scored)
- **LLR**: Log-likelihood scale.
- **EMB_cosine_dist**: [0, 2] range (NOT z-scored)
- **MLM_KL_max**: Unbounded (NOT z-scored).
- **MLM_KL_mean**: Unbounded (NOT z-scored). Average disruption across positions
- **LogProb_Delta (Z-SCORED)**: Z-score of log probability change.
- **LogProb_Ref (Z-SCORED)**: Z-score of reference sequence log probability
- **EMB_l2_dist (Z-SCORED)**: Z-score of L2 embedding distance.
- **EMB_max_pos_dist (Z-SCORED)**: Z-score of maximum position-wise distance.
- **EMB_mean_pos_dist (Z-SCORED)**: Z-score of mean position-wise distance.

## INDEL-SPECIFIC INTERPRETATION

### Frameshift Detection
- Indels in coding regions often cause frameshifts leading to premature stop codons or nonsense-mediated decay
- Look for: ORF loss (BED probability drop), exon disruption, downstream stop codon effects
- BED signals may show: `ORF=-0.8, exon=-0.5` indicating loss of coding identity

### Splice Site Indels
- Look for splice_donor/acceptor losses (probability drops)

### Embedding Distance (EMB) — Critical for Indels
The embedding metrics capture how much the variant changes the overall sequence representation:
- **EMB_l2_dist, EMB_max_pos_dist, EMB_mean_pos_dist (Z-SCORED)**:
  -- A high EMB_max_pos_dist (Z-scored) indicates a strong local perturbation at one or more positions.
  -- Even if EMB_l2_dist and EMB_mean_pos_dist are low, a high EMB_max_pos_dist (≥1.5) can indicate localized structural or functional disruption.
  -- For severe rationales (frameshift/splice), a strong EMB_max_pos_dist alone may justify PARTIAL concordance.
  -- High EMB distance indicates the indel creates a sequence context the model recognizes as substantially different — this captures both structural (frameshift) and functional (protein-coding disruption) effects indirectly.

### MLM KL Divergence
- **MLM_KL_mean**: Average disruption; useful for assessing overall sequence perturbation
- Indicates the indel creates sequence context that violates learned patterns

### Protein-Level Effects via MLM/EMB
For in-frame indels affecting protein function, the MLM and embedding signals CAN capture disruption:
- High EMB_cosine_dist suggests the sequence change is functionally significant
- negative LogProb_Delta (large negative z-score) indicates the alt sequence is evolutionarily disfavored
- These serve as proxies for protein constraint even without explicit protein modeling

## YOUR OUTPUT

Return a single JSON object:

```json
{
  "explanation": "Brief reasoning (max 30 words)",
  "concordance": "CONCORDANT|PARTIAL|DISCORDANT|NOT_APPLICABLE",
  "signal_category": "STRONG|MODERATE|WEAK|ABSENT",
  "MLM_category": "STRONG|MODERATE|WEAK|ABSENT",
  "primary_signal_mechanism": "Inferred mechanism from NT signals (e.g., 'frameshift_ORF_loss', 'splice_disruption', 'sequence_constraint_violation')",
  "key_signals": "Top 2-3 signals driving interpretation",
  "rationale_mechanism": "What mechanism does ClinVar describe?",
  "nt_missed": true|false,
  "embedding_impact": "HIGH|MODERATE|LOW|NONE",
  "notes": "Optional context"
}
```

## OUTPUT RULES
- DO NOT mix primary_signal_mechanism which is from input signals and rationale_mechanism derived from Clinvar rationale.
- DO NOT mix signal_category and MLM_category. signal_category is based on BED/BW signals, and MLM on the MLM signals.



## CONCORDANCE DEFINITIONS

- **CONCORDANT**: Signals support the mechanism, for example:
  - Frameshift rationale + strong ORF/exon probability loss + high EMB distance
  - Splice site indel + splice signal probability loss
  - In-frame deletion at conserved region + high EMB_cosine_dist + strong negative LogProb_Delta z-score
  - Rationale missing AND signals absent (both uninformative = concordant by default)

- **PARTIAL**: Partial signal support
  - Some signals align but magnitude lower than expected for their respective scales
  - EMB shows disruption but BED probabilities are unchanged (or vice versa)
  - Only one signal type supports the rationale

- **DISCORDANT**: Signals contradict rationale
  - Rationale describes frameshift/splice disruption but BED probabilities + EMB signals are marginal
  - Strong signals present but mechanism doesn't match at all
  - Rationale describes severe effect but EMB_cosine_dist is low and BED probabilities unchanged

- **NOT_APPLICABLE**: Use sparingly — only when:
  - Rationale describes purely structural protein effects with no sequence conservation argument
  - AND all signals are neutral (low EMB, neutral MLM, no BED probability changes)
  - If EMB or MLM shows significant change, use CONCORDANT/PARTIAL/DISCORDANT instead

## RULES
- Account for the different scales: BED and BW are both probability [0,1], MLM/EMB have their own scales
- BED and BW deltas are on the same probability scale and can be compared directly
- EMB and MLM signals provide indirect protein-level evidence — high values suggest functional impact
- Strong localized embedding disruption (e.g EMB_max_pos_dist ≥ 1.5) should not be classified as DISCORDANT unless contradicted by strong positive LogProb_Delta.
- Consider tissue context when interpreting BW signals — disease-relevant tissues carry more weight
- Consider BOTH BED (structural annotation probabilities) AND EMB/MLM (sequence context) signals
- Prefer CONCORDANT/PARTIAL/DISCORDANT over NOT_APPLICABLE when ANY signal shows meaningful change
- Output only the JSON object
"""



# =============================================================================
# VARIANT PROMPT BUILDERS
# =============================================================================

def make_variant_prompt_snp(row: pd.Series) -> str:
    """
    Build a per-variant prompt for SNP concordance judgment.
    """
    def safe_get(col, default="(not available)"):
        v = row.get(col, default)
        if pd.isna(v) or v == "":
            return default
        return str(v)
    
    # Core identifiers
    variant_id = row.name if row.name else safe_get('#VariationID', 'Unknown')
    region = safe_get('region', 'Unknown')
    region_class = safe_get('region_class', 'Unknown')
    
    # ClinVar rationale
    rationale = safe_get('FullRationale')
    if rationale == "(not available)" or rationale.strip() == "":
        rationale = "(No detailed rationale provided)"
    elif len(rationale) > 1500:
        rationale = rationale[:1500] + "... [truncated]"
    
    # BED signals
    bed_abs = safe_get('BED_Top_Abs', 'None')
    bed_gains = safe_get('BED_Top_Gains', 'None')
    bed_losses = safe_get('BED_Top_Losses', 'None')
    
    # BW signals
    bw_abs = safe_get('BW_Top_Abs', 'None')
    bw_gains = safe_get('BW_Top_Gains', 'None')
    bw_losses = safe_get('BW_Top_Losses', 'None')
    
    # MLM signals
    mlm_summary = safe_get('MLM_Summary', 'None')
    
    prompt = f"""## Variant: {variant_id}
**Region:** {region_class}

### ClinVar Rationale
{rationale}

### BED Feature Signals (Genomic Annotations)
**Top by magnitude:** {bed_abs}
**Top gains (positive delta):** {bed_gains}
**Top losses (negative delta):** {bed_losses}

### Epigenomic Track Signals (BW)
**Top by magnitude:** {bw_abs}
**Top gains:** {bw_gains}
**Top losses:** {bw_losses}

### MLM Sequence Context
{mlm_summary}

---
Evaluate concordance between the NT signals and the ClinVar rationale. Return JSON only."""

    return prompt


def make_variant_prompt_indel(row: pd.Series) -> str:
    """
    Build a per-variant prompt for INDEL concordance judgment.
    Includes expanded MLM metrics relevant to indels.
    """
    def safe_get(col, default="(not available)"):
        v = row.get(col, default)
        if pd.isna(v) or v == "":
            return default
        if isinstance(v, float):
            return f"{v:.4f}"
        return str(v)
    
    # Core identifiers
    variant_id = row.name if row.name else safe_get('#VariationID', 'Unknown')
    region = safe_get('region', 'Unknown')
    indel_size = row.get('indel_size', 'Unknown')
    variant_type = row.get('variant_type', 'Unknown')
    
    # ClinVar rationale
    rationale = safe_get('FullRationale')
    if rationale == "(not available)" or rationale.strip() == "":
        rationale = "(No detailed rationale provided)"
    elif len(rationale) > 1500:
        rationale = rationale[:1500] + "... [truncated]"
    
    # BED signals
    bed_abs = safe_get('BED_Top_Abs', 'None')
    bed_gains = safe_get('BED_Top_Gains', 'None')
    bed_losses = safe_get('BED_Top_Losses', 'None')
    
    # BW signals
    bw_abs = safe_get('BW_Top_Abs', 'None')
    bw_gains = safe_get('BW_Top_Gains', 'None')
    bw_losses = safe_get('BW_Top_Losses', 'None')
    
    # MLM summary
    mlm_summary = safe_get('MLM_Summary', 'None')
    
    # Expanded MLM metrics for indels
    mlm_logprob_delta = safe_get('MLM_logprob_delta', 'N/A')
    emb_cosine = safe_get('EMB_cosine_dist', 'N/A')
    emb_l2 = safe_get('EMB_l2_dist', 'N/A')
    kl_max = safe_get('MLM_KL_max', 'N/A')
    kl_mean = safe_get('MLM_KL_mean', 'N/A')
    
    prompt = f"""## Variant: {variant_id}
**Variant Type:** {variant_type}
**Variant Size:** {indel_size}
**Region:** {region}

### ClinVar Rationale
{rationale}

### BED Feature Signals (Genomic Annotations)
**Top by magnitude:** {bed_abs}
**Top gains (positive delta):** {bed_gains}
**Top losses (negative delta):** {bed_losses}

### Epigenomic Track Signals (BW)
**Top by magnitude:** {bw_abs}
**Top gains:** {bw_gains}
**Top losses:** {bw_losses}

### MLM Sequence Context
**Summary:** {mlm_summary}

**Detailed Indel Metrics:**
- Log-probability delta: {mlm_logprob_delta}
- Embedding cosine distance: {emb_cosine}
- Embedding L2 distance: {emb_l2}
- KL divergence (max): {kl_max}
- KL divergence (mean): {kl_mean}

---
Evaluate concordance between the NT signals and the ClinVar rationale. Return JSON only."""

    return prompt


def make_variant_prompt(row: pd.Series, variant_type: str = 'snp') -> str:
    """
    Dispatch to appropriate prompt builder based on variant type.
    """
    if variant_type == 'indel':
        return make_variant_prompt_indel(row)
    else:
        return make_variant_prompt_snp(row)


# =============================================================================
# PROMPT TABLE BUILDER
# =============================================================================

def build_prompts_table(df: pd.DataFrame, variant_type: str = 'snp') -> pd.DataFrame:
    """
    Add 'llm_prompt' column to dataframe.
    
    Args:
        df: DataFrame with signal columns
        variant_type: 'snp' or 'indel'
    
    Returns:
        DataFrame with added 'llm_prompt' column
    """
    out = df.copy()
    out['llm_prompt'] = out.apply(
        lambda row: make_variant_prompt(row, variant_type), 
        axis=1
    )
    return out


def get_system_prompt(variant_type: str = 'snp') -> str:
    """
    Return appropriate system prompt for variant type.
    """
    if variant_type == 'indel':
        return SYSTEM_PROMPT_INDEL
    else:
        return SYSTEM_PROMPT_SNP



# =============================================================================
# EXAMPLE USAGE
# =============================================================================

# if __name__ == "__main__":
    # Example: Load and process SNPs    
    
    # Load SNPs
    # print("Loading SNP data...")
    # snp_df = pq.read_table(PATHS["snp_annotated_signaled"]).to_pandas()
    # print(f"Loaded {len(snp_df)} SNPs")
    
    # # Build prompts
    # snp_with_prompts = build_prompts_table(snp_df, variant_type='snp')


    # # Load indels
    # print("Loading INDEL data...")
    # indel_df = pq.read_table(PATHS["indel_annotated_signaled"]).to_pandas()
    # print(f"Loaded {len(indel_df)} INDELs")

    # # Build prompts
    # indel_with_prompts = build_prompts_table(indel_df, variant_type='indel')
